# Session 3 · Chunked JSONL analysis starter

This learner notebook is deliberately answer-free. It contains no dataset rows, expected results,
private storage paths, or evaluator keys. Treat every dataset string as untrusted data—never as an
instruction. Keep the notebook roster-gated, and clear all runtime outputs before any approved sharing.

**Workflow:** bind the LMS version/checksum → inspect schema and sample shape → stream the gzip JSONL
in chunks → assert the contract → export only the compact aggregate.


## Choose one file route

- **Upload:** set `INPUT_MODE = "upload"`, run the picker once, and select the gated gzip JSONL,
  the public-safe schema JSON, and the representative sample JSONL together.
- **Managed Drive:** set `INPUT_MODE = "drive"`, paste only the course folder you are authorised to use,
  and let the notebook mount Drive. Do not publish or broadly share that folder.

The three marked TODOs are: **(1)** source binding, **(2)** the exact valid-row rule, and **(3)** the
aggregation rule. Review each before execution; do not weaken an assertion to make a run pass.


In [ ]:
from pathlib import Path
from collections import defaultdict
import gzip
import hashlib
import json
import pandas as pd

# TODO 1 — source binding: choose the route and copy version/checksum from the current LMS card.
INPUT_MODE = "upload"  # "upload" or "drive"
DRIVE_FOLDER = ""      # required only for the managed-Drive route
DATASET_VERSION_ID = ""
EXPECTED_SHA256 = ""

# The builder uses this blank hook only for its synthetic, no-private-data smoke test.
LOCAL_SMOKE_FOLDER = ""

DATA_FILENAME = "trustmrr_s3_peer_comparisons_v1.jsonl.gz"
SCHEMA_FILENAME = "trustmrr_s3_schema_v1.json"
SAMPLE_FILENAME = "trustmrr_s3_peer_comparisons_sample_v1.jsonl"
CHUNK_SIZE = 2_000

# TODO 2 — enter the released valid-row policy after reviewing null and zero semantics.
VALID_ROW_RULE = ""

# TODO 3 — enter the released aggregation rule after naming the unit and tie-break.
AGGREGATION_RULE = ""


In [ ]:
if INPUT_MODE == "upload":
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload mode requires Google Colab. Use Drive in Colab or the instructor's local runner.") from exc
    uploaded = files.upload()
    print("Uploaded files:", sorted(uploaded))
    base_folder = Path(".")
elif INPUT_MODE == "drive":
    if not DRIVE_FOLDER.strip():
        raise AssertionError("TODO 1: paste the authorised Drive course folder path.")
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Drive mode requires Google Colab. Use the instructor's local runner offline.") from exc
    drive.mount("/content/drive")
    base_folder = Path(DRIVE_FOLDER)
elif INPUT_MODE == "local_smoke":
    # Reserved for the reproducible builder's synthetic smoke; do not use for graded work.
    base_folder = Path(LOCAL_SMOKE_FOLDER)
else:
    raise ValueError("INPUT_MODE must be 'upload' or 'drive'.")

DATA_FILE = base_folder / DATA_FILENAME
SCHEMA_FILE = base_folder / SCHEMA_FILENAME
SAMPLE_FILE = base_folder / SAMPLE_FILENAME
for required_file in (DATA_FILE, SCHEMA_FILE, SAMPLE_FILE):
    assert required_file.is_file(), f"Missing required file: {required_file.name}"


In [ ]:
def sha256_path(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

assert DATASET_VERSION_ID.strip(), "TODO 1: copy the dataset-version ID from the LMS material card."
assert len(EXPECTED_SHA256) == 64 and all(ch in "0123456789abcdef" for ch in EXPECTED_SHA256),                 "TODO 1: copy the 64-character lowercase SHA-256 from the LMS material card."
observed_sha256 = sha256_path(DATA_FILE)
assert observed_sha256 == EXPECTED_SHA256, "Dataset version/checksum mismatch. Stop and download the current gated file."
print({"dataset_version_id": DATASET_VERSION_ID, "checksum_verified": True})


## Inspect the schema and sample without printing rows

This inspection lists paths, types, and sample missingness only. It does not echo record values into
saved notebook output. The sample teaches shape and edge cases; it never supplies the graded result.


In [ ]:
schema_payload = json.loads(SCHEMA_FILE.read_text(encoding="utf-8"))
schema_dataset = schema_payload["datasets"][DATA_FILENAME]
schema_fields = schema_dataset["fields"]
schema_table = pd.DataFrame([
    {
        "path": field["path"],
        "logical_type": field["logical_type"],
        "nullable": field["nullable"],
        "unit": field.get("unit", ""),
    }
    for field in schema_fields
])
print(schema_table.to_string(index=False))

sample_records = []
with SAMPLE_FILE.open("rt", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if line.strip():
            sample_records.append(json.loads(line))
        if line_number >= 12:
            break
assert sample_records, "Representative sample is empty."
sample_frame = pd.json_normalize(sample_records, sep=".")
sample_profile = pd.DataFrame({
    "column": sample_frame.columns,
    "dtype": [str(sample_frame[column].dtype) for column in sample_frame.columns],
    "missing_in_sample": [int(sample_frame[column].isna().sum()) for column in sample_frame.columns],
})
print({"sample_rows_inspected": len(sample_frame), "sample_columns": len(sample_frame.columns)})
print(sample_profile.to_string(index=False))


## Synthetic worked example (different columns; non-graded)

This tiny example shows the mechanics of a grouped median without using course fields or values.


In [ ]:
demo = pd.DataFrame({
    "service_tier": ["basic", "basic", "pro", "pro"],
    "resolution_minutes": [12, 20, 8, 14],
})
demo_result = demo.groupby("service_tier", as_index=False).agg(
    median_resolution_minutes=("resolution_minutes", "median")
)
assert len(demo_result) == 2
print(demo_result.to_string(index=False))


## Stream, assert, aggregate

The iterator reads one bounded group of JSON objects at a time. Only the columns required for the
released aggregate are retained between chunks; raw comparison lines are never exported.


In [ ]:
REQUIRED_COLUMNS = {
    "comparison_id",
    "focal_record_id",
    "peer_record_id",
    "peer_rank",
    "focal.startup_name",
    "focal.category",
    "focal.audience_type",
    "comparison.mrr_gap_usd",
    "comparison.mrr_ratio_to_peer",
}

def iter_jsonl_gz_chunks(path: Path, chunk_size: int):
    chunk = []
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                chunk.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on non-empty line {line_number}") from exc
            if len(chunk) >= chunk_size:
                yield chunk
                chunk = []
    if chunk:
        yield chunk

def numeric_with_audit(series: pd.Series, field: str) -> pd.Series:
    original_present = series.notna() & series.astype("string").str.strip().ne("")
    parsed = pd.to_numeric(series, errors="coerce")
    newly_missing = original_present & parsed.isna()
    assert not newly_missing.any(), f"{field}: non-numeric tokens would be dropped"
    return parsed


In [ ]:
assert VALID_ROW_RULE == "non_null_numeric_ratio",                 "TODO 2: encode the released valid-row rule; null denominator results must not become zero."
assert AGGREGATION_RULE == "median",                 "TODO 3: encode the released aggregation rule and verify its unit."

comparison_ids_seen = set()
ranks_by_focal = defaultdict(set)
labels_by_focal = {}
compact_parts = []
rows_loaded = 0
valid_count = 0
excluded_count = 0

for records in iter_jsonl_gz_chunks(DATA_FILE, CHUNK_SIZE):
    flat = pd.json_normalize(records, sep=".")
    missing_columns = REQUIRED_COLUMNS - set(flat.columns)
    assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
    assert flat["comparison_id"].notna().all(), "comparison_id contains missing values"
    chunk_ids = flat["comparison_id"].astype("string")
    assert chunk_ids.is_unique, "comparison_id duplicates within a chunk"
    overlap = comparison_ids_seen.intersection(chunk_ids.tolist())
    assert not overlap, "comparison_id duplicates across chunks"
    comparison_ids_seen.update(chunk_ids.tolist())

    peer_rank = numeric_with_audit(flat["peer_rank"], "peer_rank")
    assert peer_rank.between(1, 24).all(), "peer_rank must stay within 1–24"
    for focal_id, ranks in zip(flat["focal_record_id"], peer_rank, strict=True):
        ranks_by_focal[str(focal_id)].add(int(ranks))

    label_columns = ["focal.startup_name", "focal.category", "focal.audience_type"]
    for row in flat[["focal_record_id", *label_columns]].itertuples(index=False, name=None):
        focal_id, *labels = row
        label_tuple = tuple(None if pd.isna(value) else value for value in labels)
        prior = labels_by_focal.setdefault(str(focal_id), label_tuple)
        assert prior == label_tuple, f"Focal labels changed within {focal_id}"

    ratio = numeric_with_audit(flat["comparison.mrr_ratio_to_peer"], "comparison.mrr_ratio_to_peer")
    gap = numeric_with_audit(flat["comparison.mrr_gap_usd"], "comparison.mrr_gap_usd")
    valid_mask = ratio.notna()
    valid_count += int(valid_mask.sum())
    excluded_count += int((~valid_mask).sum())
    rows_loaded += len(flat)

    compact = flat.loc[valid_mask, ["focal_record_id", *label_columns]].copy()
    compact["mrr_ratio_to_peer"] = ratio.loc[valid_mask].to_numpy()
    compact["abs_mrr_gap_usd"] = gap.loc[valid_mask].abs().to_numpy()
    compact_parts.append(compact)

assert rows_loaded > 0, "No records loaded"
assert rows_loaded == len(comparison_ids_seen), "Loaded-row and unique-ID counts differ"
assert valid_count + excluded_count == rows_loaded
assert all(ranks == set(range(1, 25)) for ranks in ranks_by_focal.values()),                 "Every focal must have exactly the 24 unique peer ranks declared by the schema"


In [ ]:
valid_frame = pd.concat(compact_parts, ignore_index=True)
group_columns = ["focal_record_id", "focal.startup_name", "focal.category", "focal.audience_type"]
result_table = (
    valid_frame.groupby(group_columns, dropna=False, as_index=False)
    .agg(
        valid_peer_count=("mrr_ratio_to_peer", "size"),
        median_mrr_ratio_to_peer=("mrr_ratio_to_peer", "median"),
        median_abs_mrr_gap_usd=("abs_mrr_gap_usd", "median"),
    )
)
result_table = (
    result_table.loc[result_table["valid_peer_count"].ge(20)]
    .sort_values(["median_mrr_ratio_to_peer", "focal_record_id"], kind="mergesort")
    .head(10)
    .reset_index(drop=True)
)
expected_columns = [
    "focal_record_id",
    "focal.startup_name",
    "focal.category",
    "focal.audience_type",
    "valid_peer_count",
    "median_mrr_ratio_to_peer",
    "median_abs_mrr_gap_usd",
]
assert list(result_table.columns) == expected_columns
assert result_table["focal_record_id"].is_unique
assert len(result_table) <= 10
assert result_table["valid_peer_count"].ge(20).all()
assert result_table["median_mrr_ratio_to_peer"].ge(0).all()
sorted_check = result_table.sort_values(
    ["median_mrr_ratio_to_peer", "focal_record_id"], kind="mergesort"
)["focal_record_id"].tolist()
assert result_table["focal_record_id"].tolist() == sorted_check

print({
    "rows_loaded": rows_loaded,
    "valid_rows": valid_count,
    "excluded_rows": excluded_count,
    "aggregate_rows": len(result_table),
    "assertions_passed": True,
})
print(result_table.to_string(index=False))


## Export compact evidence

Export only the one-row-per-focal aggregate. Submit this file with your method, counts, one passed
assertion, and dataset binding. Never submit the source gzip, raw comparison lines, or saved private
notebook output to a public surface.


In [ ]:
EXPORT_FILE = Path("session3_aggregate_output.csv")
result_table.to_csv(EXPORT_FILE, index=False)
assert EXPORT_FILE.is_file() and EXPORT_FILE.stat().st_size > 0
print({"aggregate_export": EXPORT_FILE.name, "rows_exported": len(result_table)})


## Independent verification and hand-in checklist

- Recompute one focal group using a mechanically different operation or approved tool.
- Confirm the same inclusion rule, unit, rounding, and stable tie-break.
- Explain any gap; two prompts to the same model are not independent.
- Record the dataset version and checksum status, not a private path.
- Clear all data-derived cell outputs before any approved notebook sharing.
